In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import json
import gc

folder_path = '/content/drive/MyDrive/BT4222_Group11'
eng_path = f'{folder_path}/engineered data'

print("1. Loading Mappings...")
with open(f'{folder_path}/anime_mapping.json', 'r') as f:
    anime_mapping = json.load(f)
num_anime = len(anime_mapping)
id_to_idx = {int(k): v for k, v in anime_mapping.items()}

print("2. Loading Xin Wei's Data & Calculating Dimensions...")

df_xinwei = pd.read_csv(f'{folder_path}/anime_complete_encoded.csv', low_memory=False)
id_col_x = 'MAL_ID' if 'MAL_ID' in df_xinwei.columns else 'anime_id'
df_xinwei = df_xinwei.set_index(id_col_x).select_dtypes(include=[np.number]).astype(np.float32)
xinwei_dim = df_xinwei.shape[1]

# Yichuan
yichuan_dim = np.load(f'{eng_path}/synopsis_bert.npy', mmap_mode='r').shape[1]

# Yuran
yuran_dim = np.load(f'{eng_path}/review_features/review_bert_embeddings.npy', mmap_mode='r').shape[1]

total_dim = xinwei_dim + yichuan_dim + yuran_dim
print(f"   -> Xin Wei features: {xinwei_dim}")
print(f"   -> Yichuan features: {yichuan_dim}")
print(f"   -> Yuran features: {yuran_dim}")
print(f"Total Features per Anime: {total_dim}")

print("\n3. Pre-allocating the Master Matrix in RAM...")
# float32 saves 50% of your RAM!
master_matrix = np.zeros((num_anime, total_dim), dtype=np.float32)

# XIN WEI
print("\n4. Slotting Xin Wei's data...")
for raw_id, row in df_xinwei.iterrows():
    if raw_id in id_to_idx:
        master_matrix[id_to_idx[raw_id], 0:xinwei_dim] = row.values

del df_xinwei
gc.collect() # Instantly clear RAM
print("   -> Xin Wei's data slotted and RAM cleared.")

#YICHUAN
print("\n5. Loading and slotting Yichuan's data...")
yichuan_embs = np.load(f'{eng_path}/synopsis_bert.npy').astype(np.float32)

df_base_ids = pd.read_csv(f'{folder_path}/anime_complete.csv', usecols=['MAL_ID'])

for i, raw_id in enumerate(df_base_ids['MAL_ID']):
    if raw_id in id_to_idx:
        master_matrix[id_to_idx[raw_id], xinwei_dim : xinwei_dim+yichuan_dim] = yichuan_embs[i]

del yichuan_embs, df_base_ids
gc.collect() # Instantly clear RAM
print("   -> Yichuan's data slotted and RAM cleared.")

#YURAN
print("\n6. Loading and slotting Yuran's data...")
review_embs = np.load(f'{eng_path}/review_features/review_bert_embeddings.npy').astype(np.float32)
review_ids = np.load(f'{eng_path}/review_features/review_anime_ids.npy')

for i, raw_id in enumerate(review_ids):
    raw_id = int(raw_id)
    if raw_id in id_to_idx:
        master_matrix[id_to_idx[raw_id], xinwei_dim+yichuan_dim : ] = review_embs[i]

del review_embs, review_ids
gc.collect()
print("   -> Yuran's data slotted and RAM cleared.")

print("\n7. Saving Master Matrix to Google Drive...")
save_path = f'{folder_path}/MASTER_ANIME_TOWER_FEATURES.npy'
np.save(save_path, master_matrix)
print(f"✅✅✅ SUCCESS! Final Master Matrix (Shape: {master_matrix.shape}) is ready for Caleb!")

1. Loading Mappings...
2. Loading Xin Wei's Data & Calculating Dimensions...
   -> Xin Wei features: 65
   -> Yichuan features: 384
   -> Yuran features: 384
Total Features per Anime: 833

3. Pre-allocating the Master Matrix in RAM...

4. Slotting Xin Wei's data...
   -> Xin Wei's data slotted and RAM cleared.

5. Loading and slotting Yichuan's data...
   -> Yichuan's data slotted and RAM cleared.

6. Loading and slotting Yuran's data...
   -> Yuran's data slotted and RAM cleared.

7. Saving Master Matrix to Google Drive...
✅✅✅ SUCCESS! Final Master Matrix (Shape: (17562, 833)) is ready for Caleb!


In [3]:

import pandas as pd
import numpy as np
import json
import gc

folder_path = '/content/drive/MyDrive/BT4222_Group11'
eng_path = f'{folder_path}/engineered_data'

print("1. Loading Caleb's 1-Based Mappings...")
with open(f'{folder_path}/new_splits/anime_id_map.json', 'r') as f:
    anime_mapping = json.load(f)

matrix_rows = max(anime_mapping.values()) + 1
id_to_idx = {int(k): int(v) for k, v in anime_mapping.items()}

print("2. Processing Xin Wei's Data (Dropping noisy columns)...")
df_xinwei = pd.read_csv(f'{folder_path}/anime_complete_encoded.csv', low_memory=False)

# Aggressive filtering
noisy_keywords = ['name', 'producer', 'licensor', 'aired', 'character', 'staff', 'synopsis']
id_col_x = 'MAL_ID' if 'MAL_ID' in df_xinwei.columns else 'anime_id'

cols_to_keep = []
for col in df_xinwei.columns:
    if col == id_col_x:
        cols_to_keep.append(col)
        continue
    if any(keyword in col.lower() for keyword in noisy_keywords):
        continue
    cols_to_keep.append(col)

df_xinwei = df_xinwei[cols_to_keep]
df_xinwei = df_xinwei.set_index(id_col_x).select_dtypes(include=[np.number]).astype(np.float32)
xinwei_dim = df_xinwei.shape[1]

yichuan_dim = np.load(f'{eng_path}/synopsis_bert.npy', mmap_mode='r').shape[1]

total_dim = xinwei_dim + yichuan_dim
print(f"   -> Lean Xin Wei features: {xinwei_dim}")
print(f"   -> Yichuan features: {yichuan_dim}")
print(f"Total Lean Features per Anime: {total_dim}")

print(f"\n3. Pre-allocating the LEAN Matrix (Shape: {matrix_rows} rows, {total_dim} features)...")
master_matrix = np.zeros((matrix_rows, total_dim), dtype=np.float32)

print("4. Slotting Lean Xin Wei data...")
for raw_id, row in df_xinwei.iterrows():
    if raw_id in id_to_idx:
        master_matrix[id_to_idx[raw_id], 0:xinwei_dim] = row.values
del df_xinwei; gc.collect()

print("5. Slotting Yichuan's data...")
yichuan_embs = np.load(f'{eng_path}/synopsis_bert.npy').astype(np.float32)
df_base_ids = pd.read_csv(f'{folder_path}/anime_complete.csv', usecols=['MAL_ID'])

for i, raw_id in enumerate(df_base_ids['MAL_ID']):
    if raw_id in id_to_idx:
        master_matrix[id_to_idx[raw_id], xinwei_dim : xinwei_dim+yichuan_dim] = yichuan_embs[i]
del yichuan_embs, df_base_ids; gc.collect()

print("\n6. Saving LEAN 1-Based Master Matrix...")
save_path = f'{folder_path}/anime_tower_1_based.npy'
np.save(save_path, master_matrix)
print(f"✅ SUCCESS! New Lean Matrix shape is {master_matrix.shape}. Row 0 is perfectly blank for padding!")

1. Loading Caleb's 1-Based Mappings...
2. Processing Xin Wei's Data (Dropping noisy columns)...
   -> Lean Xin Wei features: 65
   -> Yichuan features: 384
Total Lean Features per Anime: 449

3. Pre-allocating the LEAN Matrix (Shape: 13010 rows, 449 features)...
4. Slotting Lean Xin Wei data...
5. Slotting Yichuan's data...

6. Saving LEAN 1-Based Master Matrix...
✅ SUCCESS! New Lean Matrix shape is (13010, 449). Row 0 is perfectly blank for padding!
